# 07 — CNN rung 3: full nested CV, the gate decision

**Decision this feeds** (`docs/superpowers/specs/2026-09-09-cnn-train-rung23-design.md`):
the actual gate — does the CNN beat `model.build_combat_baseline()`
(0.5290 log loss) by more than noise? 5-fold CV, repeated 5x (varying
both the model-init seed and the fold-split seed together, per
Bouthillier et al. 2021 — `RESOURCES.md`), nested per fold (an inner
90/10 split picks the early-stopping epoch; the outer fold is never used
for training or stopping — see notebook 06's intro for why). Gate rule,
decided in the spec *before* seeing this run's numbers: the CNN
graduates only if (a) a paired bootstrap 95% CI on the delta vs. 0.5290
excludes zero, **and** (b) the 5-repeat mean beats 0.5290 by more than 2x
the larger of the two runs' standard deviations.

**Batch size / LR**: uses the winner from `notebooks/06_cnn_rung2.ipynb`'s
mini-experiment — update the `batch_size, lr = ...` line below if that
notebook's winner differs from the placeholder here.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, not any per-row output.

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels. Rebuilds (or
# reuses, if this session is still warm from notebooks/06) the shared
# on-disk volume cache.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache ready: {time.time() - cache_start:.1f}s for {len(uids)} volumes "
      f"(0s if notebook 06 already built it this session).")

In [ ]:
# [RUN ME] (no data access itself). num_workers=0 always -- see notebook
# 06's cell 2 for why.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    """See notebooks/06_cnn_rung2.ipynb's identical helper for the full
    docstring."""
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x (model-init seed and
# fold-split seed varied together, per Bouthillier et al. 2021 --
# RESOURCES.md) for the rung-3 gate decision.
batch_size, lr = 32, 2e-3  # UPDATE to notebook 06's actual winner if different
N_REPEATS = 5
oof_repeats = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        # NOTE: seed=config.SEED, fold 0 here repeats notebooks/06's
        # winning-candidate fold-0 training almost exactly (same split,
        # same seed) -- ~30s of GPU at a warm cache, not worth the added
        # complexity of passing state between notebook sessions to avoid.
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung3_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung3_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats])
print(f"\n{N_REPEATS}-repeat CNN pooled log loss: mean={repeat_scores.mean():.4f}, sd={repeat_scores.std():.4f}")
print("classical build_combat_baseline() (notebooks/04, README.md): 0.5290")

In [ ]:
# [RUN ME] -- (re)compute build_combat_baseline()'s OOF predictions on the
# SAME fold split as the CNN's first CV repeat (seed=config.SEED), so the
# paired bootstrap below compares the two models row-for-row on identical
# splits. CPU-only, seconds.
baseline_feat_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")
baseline_feat_df = baseline_feat_df.set_index(config.UID_COLUMN).loc[uids].reset_index()
baseline_X = baseline_feat_df[["abs_asym", "striatal_ratio"]].to_numpy()
baseline_y = baseline_feat_df[config.TARGET_COLUMN].to_numpy()
baseline_family = baseline_feat_df["inplane_family"].to_numpy()

baseline_oof = np.zeros(len(uids))
baseline_folds = evaluate.make_folds(baseline_y, baseline_family, n_splits=config.N_FOLDS, random_state=config.SEED)
for train_idx, test_idx in baseline_folds:
    pipeline = model.build_combat_baseline()
    pipeline.fit(baseline_X[train_idx], baseline_y[train_idx], baseline_family[train_idx])
    baseline_oof[test_idx] = pipeline.predict_proba(baseline_X[test_idx], baseline_family[test_idx])[:, 1]

baseline_pooled_logloss = evaluate.log_loss_score(baseline_y, baseline_oof)
print(f"baseline OOF pooled log loss (this split, seed={config.SEED}): {baseline_pooled_logloss:.4f} "
      f"(RESOURCES.md/README.md's recorded 5-seed mean: 0.5290)")
np.save(config.DATA_PROCESSED / "baseline_oof_seed_match.npy", baseline_oof)

In [ ]:
# [RUN ME] -- paired bootstrap (Varoquaux 2018 -- RESOURCES.md) on the
# CNN's first-repeat OOF vs. the same-split baseline OOF, plus the
# rung-3 gate rule decided in the spec BEFORE seeing this number.
CLASSICAL_BASELINE_LOGLOSS = 0.5290  # build_combat_baseline(), README.md, 5-seed mean
BASELINE_SD_RECORDED = 0.0011  # notebooks/04's recorded per-variant sd upper bound

cnn_oof_for_pairing = oof_repeats[0]  # same seed/split as baseline_oof above
y_true = np.array(labels)

rng = np.random.RandomState(config.SEED)
n = len(y_true)
n_bootstrap = 1000
deltas = np.empty(n_bootstrap)
for b in range(n_bootstrap):
    idx = rng.randint(0, n, size=n)
    cnn_ll = evaluate.log_loss_score(y_true[idx], cnn_oof_for_pairing[idx])
    baseline_ll = evaluate.log_loss_score(y_true[idx], baseline_oof[idx])
    deltas[b] = cnn_ll - baseline_ll  # negative = CNN better

ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
ci_excludes_zero_favoring_cnn = ci_high < 0

repeat_mean = repeat_scores.mean()
repeat_sd = repeat_scores.std(ddof=1)
noise_threshold = 2 * max(repeat_sd, BASELINE_SD_RECORDED)
mean_beats_baseline_by = CLASSICAL_BASELINE_LOGLOSS - repeat_mean

gate_passed = ci_excludes_zero_favoring_cnn and (mean_beats_baseline_by > noise_threshold)

print(f"paired bootstrap delta (CNN - baseline), 95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"5-repeat mean={repeat_mean:.4f}, sd={repeat_sd:.4f}; "
      f"beats {CLASSICAL_BASELINE_LOGLOSS} by {mean_beats_baseline_by:+.4f} "
      f"(2x max-sd noise threshold = {noise_threshold:.4f})")
print(f"GATE {'PASSED' if gate_passed else 'NOT PASSED'}: "
      f"{'CNN graduates past rung 3.' if gate_passed else 'fallback applies -- see spec Definition of done.'}")

family_arr = np.array(families)
print("\nper-family CNN log loss (first repeat's pooled OOF):")
for fam in sorted(set(families)):
    mask = family_arr == fam
    if mask.sum() < 5:
        continue
    print(f"  {fam:<12} n={int(mask.sum()):>4}  log loss={evaluate.log_loss_score(y_true[mask], cnn_oof_for_pairing[mask]):.4f}")

In [ ]:
# [RUN ME] -- fallback check (spec's Definition of done): a CNN+ComBat
# probability blend, evaluated regardless of the gate outcome above since
# it costs no GPU.
for w in [0.25, 0.5, 0.75]:
    blend = w * cnn_oof_for_pairing + (1 - w) * baseline_oof
    blend_ll = evaluate.log_loss_score(y_true, blend)
    print(f"blend w_cnn={w}: log loss={blend_ll:.4f}")

**What we're looking for:** does the CNN's 5-fold, 5-repeat nested-CV
pooled log loss beat `build_combat_baseline()` (0.5290) by more than
noise, per the gate rule fixed in the spec before this run?

**What we found:** *(paste: per-repeat pooled log loss and the 5-repeat
mean/sd; the paired-bootstrap 95% CI; the GATE PASSED/NOT PASSED line;
the per-family breakdown; the three blend log-loss numbers)*

**Decision / next step:** *(if the gate passed: the CNN becomes the
submission candidate -- next is submission packaging + local Docker
rehearsal, README.md's remaining Next step. If not: ship
build_combat_baseline() as the submission, note whether any blend weight
beat 0.5290 outright, and update README.md/RESOURCES.md with this
negative result, per the project's standing rule to log negative results
just like positive ones.)*